<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weekend 8 — AI Agents: AI That *Acts*
### AI Architect Mastery Program · ~2 hours · For complete beginners

## The one big idea
A normal chatbot can only **talk**. An **agent** can **do things** — look something up, do a calculation, book a slot — and then react to what happened.

**A chatbot gives you homework. An agent gives you the answer.**

## By the end you'll be able to
- Say, in one line, how an agent is different from a chatbot.
- Follow the **agent loop**: *look → think → act → check → repeat*.
- Write a simple **tool** (the thing that lets Claude act).
- Build a tiny working agent with the Claude API.
- Spot the 4 common ways agents break, and fix them.

> We use **Claude (Haiku model)** for everything. Every code cell is **real** and runs in Google Colab once you add your key. No fake/demo code.


## Setup — do this once
You need a Claude API key. In Colab, click the 🔑 (left sidebar) and add a secret named **`MY_API_KEY`** with your key. Then run the two cells below.

In [ ]:
!pip install anthropic

In [ ]:
from google.colab import userdata   # reads your Colab secret
import os, json
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()         # this is what we use to talk to Claude
MODEL = "claude-haiku-4-5-20251001"    # fast + cheap — perfect for learning
print("Ready ✅")

---
# Part 1 — Chatbot vs Agent

**A chatbot talks. An agent acts.**

Ask both: *"Is item SKU-2271 in stock at the Zepto Indiranagar store?"*
- **Chatbot:** "To check, you'd look it up in your inventory system." *(tells you what to do)*
- **Agent:** actually checks → "Yes, 3 left. That's low, I'd reorder." *(does it for you)*

### Three ways to picture it
1. **Friend on the phone vs. assistant in your office.** One gives advice; the other opens the cabinet and hands you the result.
2. **Recipe vs. cook.** One reads the recipe; the other cooks the meal.
3. **GPS voice vs. driver.** One says "turn left"; the other drives.

> **One thing to keep straight:** "the agent does it" really means **Claude decides what to do, and your code does it.** Claude never touches your systems directly. (This matters a lot later.)


### See it: a chatbot can only *describe*
With no tools, Claude can only **tell you how** to check stock — it can't actually check.

In [ ]:
reply = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": "Is SKU-2271 in stock at the Zepto Indiranagar store?"}],
)
print(reply.content[0].text)
# Notice: it explains the steps. It has no way to really look anything up.

### Where this is used (real companies)
- **Swiggy** — checks restaurant + rider availability before confirming an order.
- **Apollo Hospitals** — finds a slot and books your appointment.
- **Zepto** — watches stock and raises a reorder.

Every time, the value is in the **doing**.

### 📌 Remember this
> A chatbot **describes**. An agent **does**. And: **Claude decides, your code acts.**

### ❌ Don't mix these up
- ❌ "The agent runs the tools by itself." → ✅ Claude only *asks*; your code runs it.
- ❌ "Agents need a special model." → ✅ Same Claude. The magic is **tools + a loop**.

### 🎯 Interview question
*Q: Chatbot vs agent in one line?*
**A:** "A chatbot only writes text; an agent uses tools to take real actions and reacts to the results."

### ✅ Quick check
*Q: A bot says "Reset your password in Settings → Security." Chatbot or agent?*
**A: Chatbot** — it only describes. An agent would reset it for you.


---
# Part 2 — The Agent Loop

Every agent repeats the same 4 steps until the job is done:

```
   1. LOOK   — read the goal and the latest result
   2. THINK  — decide the next action
   3. ACT    — ask for a tool
   4. CHECK  — read what the tool gave back
        ↑________ repeat until done ________↓
```

The **thinking** is Claude. The **doing** is your code.


## How your code knows what to do next: `stop_reason`

After every reply, Claude tells you *why it stopped* in one field, `reply.stop_reason`:

| `stop_reason` | What it means | What you do |
|---|---|---|
| `"tool_use"` | Claude wants to use a tool | run the tool, send the result back, loop again |
| `"end_turn"` | Claude is finished | stop and show the answer |

**Simple rule:** while `stop_reason == "tool_use"`, keep looping. Otherwise, stop.

> **Remember:** Claude **decides**, your code **runs it**. Claude can't run your function any more than your manager can personally fix a server — it tells you what it needs, you do it and report back.


### See it
The earlier answer finished normally, so its `stop_reason` is `end_turn`.

In [ ]:
print("stop_reason:", reply.stop_reason)   # end_turn = Claude was done

### ✅ Quick check
*Q: Claude's reply has `stop_reason = "tool_use"`. What two things must your code do before asking Claude again?*
**A:** (1) Run the tool it asked for, and (2) send the result back. Then ask Claude again.


---
# Part 3 — Tools (how Claude acts)

A **tool** is just a function you let Claude use. You describe it with **three things**:

```json
{
  "name": "check_stock",
  "description": "Get how many units of an item are in stock. Use when someone asks how much is left.",
  "input_schema": {
    "type": "object",
    "properties": { "item": { "type": "string" } },
    "required": ["item"]
  }
}
```

- **name** — the label. Claude sends this back so your code knows which function to run.
- **description** — plain English: *what it does and when to use it.* **This is the most important part** — Claude reads it to pick the right tool. Vague description = wrong tool.
- **input_schema** — the inputs and their types, so Claude fills them in correctly.

Claude only ever sees this little description — never your actual code.


### See it: a tool is just a dictionary

In [ ]:
check_stock_tool = {
    "name": "check_stock",
    "description": "Get how many units of an item are in stock. Use when someone asks how much is left.",
    "input_schema": {
        "type": "object",
        "properties": {"item": {"type": "string"}},
        "required": ["item"],
    },
}
print(json.dumps(check_stock_tool, indent=2))   # this is all Claude reads

### 📌 Remember this
> A tool = **name + description + input_schema**. The **description** is what makes Claude pick the right tool.

### ❌ Don't mix these up
- ❌ "Claude picks tools by the function name." → ✅ It picks by the **description**.
- ❌ "More tools = better." → ✅ Too many similar tools confuse Claude. Keep them few and clear.

### 🎯 Interview question
*Q: Your agent keeps using the wrong tool. What do you fix first?*
**A:** The **description** — Claude chooses tools by reading it.

### ✅ Quick check
*Q: Which part tells Claude *when* to use a tool?*
**A:** The **description**.


---
# Part 4 — A Real Agent (watch it work)

Goal: *"How many units of bread should we reorder to reach 20?"*
The agent has two tools: `check_stock(item)` and `reorder_amount(current, target)`.

It will **chain** them: first find the stock (4), then work out the reorder (20 − 4 = 16).

```
check_stock(bread) → 4        (tool_use)
reorder_amount(4, 20) → 16    (tool_use)
"Reorder 16 units."           (end_turn) ✅
```


In [ ]:
# 1) the tools, as real Python functions
INVENTORY = {"milk": 12, "bread": 4, "eggs": 30}
def check_stock(item):      return INVENTORY.get(item.lower(), 0)
def reorder_amount(c, t):   return max(t - c, 0)

# 2) what Claude sees
tools = [
    {"name": "check_stock",
     "description": "Get how many units of an item are in stock.",
     "input_schema": {"type": "object",
        "properties": {"item": {"type": "string"}}, "required": ["item"]}},
    {"name": "reorder_amount",
     "description": "Work out how many units to reorder to reach a target stock level.",
     "input_schema": {"type": "object",
        "properties": {"current": {"type": "integer"}, "target": {"type": "integer"}},
        "required": ["current", "target"]}},
]

# 3) a small helper that runs whatever tool Claude asks for
def run_tool(name, args):
    if name == "check_stock":    return check_stock(args["item"])
    if name == "reorder_amount": return reorder_amount(args["current"], args["target"])

# 4) the loop
messages = [{"role": "user",
    "content": "How many units of bread should we reorder to reach 20? Use the tools."}]

for turn in range(5):                       # stop after 5 turns no matter what
    reply = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": reply.content})

    if reply.stop_reason != "tool_use":     # Claude is done
        print("\n🏁", reply.content[0].text)
        break

    results = []
    for block in reply.content:             # Claude might ask for more than one tool
        if block.type == "tool_use":
            answer = run_tool(block.name, block.input)
            print(f"🔧 {block.name}({block.input}) → {answer}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(answer)})
    messages.append({"role": "user", "content": results})   # send results back

**This chaining is the heart of agents.** No single tool could answer the question — the agent did it in steps and combined the results.

### 📌 Remember this
> Agents are powerful because they **chain** tools: the answer from one becomes the input to the next.

### 🎯 Interview question
*Q: Who runs the tool — Claude or your code?*
**A:** Your code. Claude only *asks*; your code runs it and sends the result back.


---
# Part 5 — Memory

An agent has no magic memory. It remembers by **keeping the whole conversation** in one list called `messages`.

Every turn you add to it: the goal, Claude's replies, and every tool result. You send the **whole list** back each time, so Claude can see everything that happened.

> **The #1 bug:** if you forget to send a tool's result back, Claude never sees the answer — so it asks for the **same tool again, forever.** Fix: always send the result back.


### See it: memory is just a growing list

In [ ]:
print("The agent remembers", len(messages), "messages:\n")
for i, m in enumerate(messages):
    if isinstance(m["content"], str):
        kind = "the goal (text)"
    else:
        kind = ", ".join(
            (b.get("type", "?") if isinstance(b, dict) else getattr(b, "type", "text"))
            for b in m["content"])
    print(f"  [{i}] {m['role']:9s} → {kind}")

### 📌 Remember this
> The `messages` list **is** the memory. Always add the tool result back.

### ✅ Quick check
*Q: Your agent calls a tool right, but then says "I don't have that info." Why?*
**A:** The tool's result wasn't sent back to Claude, so it never saw the answer.


---
# Part 6 — How Agents Plan

Same loop, but the agent can organise its thinking three ways:

1. **Step by step** — do one thing, look, do the next. Good for **simple** tasks. *("What's the cheapest item?" → check each price one by one.)*
2. **Break it down** — split a big goal into smaller goals. Good for **complex** tasks. *("Make the weekly restock report" → list low items → work out reorders → total the cost → format.)*
3. **Re-plan** — change the plan when something surprises you. Good for **messy** situations. *(Booking a doctor who turns out to be on leave → find the next available one.)*

> Tip: for hard plans, Claude has an **extended thinking** option — a private scratchpad to think before acting. See docs.claude.com.

### ✅ Quick check
*Q: "Book any dentist this week; if my clinic is full, try another." Which kind of planning?*
**A: Re-plan** — it changes plan when the first choice is full.


---
# Part 7 — Doing Things in Parallel

If a goal needs several **independent** lookups, Claude can ask for **several tools at once** (in one turn) instead of one at a time. That's faster.

Goal: *"Compare stock of milk, bread, and eggs."* → these don't depend on each other, so Claude asks for all three `check_stock` calls together.

Your loop already handles this — it loops over **every** tool Claude asks for, not just the first.


In [ ]:
messages2 = [{"role": "user", "content": "Compare stock of milk, bread, and eggs. Use the tools."}]
for turn in range(5):
    reply = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages2)
    messages2.append({"role": "assistant", "content": reply.content})
    if reply.stop_reason != "tool_use":
        print("\n🏁", reply.content[0].text)
        break
    results = []
    for block in reply.content:
        if block.type == "tool_use":
            answer = run_tool(block.name, block.input)
            print(f"🔧 {block.name}({block.input}) → {answer}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(answer)})
    messages2.append({"role": "user", "content": results})

> **Chain vs parallel, in one line:** chain when one result is needed for the next; go parallel when the lookups don't depend on each other.

---
# Part 8 — Why Agents Break (and the fix)

Almost every agent bug is one of these four:

| Problem | What you see | The fix |
|---|---|---|
| **Never stops** | calls tools forever | add a turn limit **and** always send the tool result back |
| **Wrong tool** | uses a tool that doesn't fit | make the **descriptions** clearer and different from each other |
| **Made-up inputs** | tool gets weird/wrong values | tighten the **input_schema** (set types, mark required) |
| **Ignores result** | answers as if the tool never ran | send the result back with the matching `tool_use_id` |

**Golden debugging trick:** print every step — the tool, the inputs, the result. Once you can *see* the loop, the bug is usually obvious. (If you see the same tool printed 5 times, the result isn't being sent back.)

### ✅ Quick check
*Q: Your agent runs forever on one tool. Most likely cause?*
**A:** The tool result isn't being sent back (and/or no turn limit).


---
# Putting it together — a real agent, simply

```
   You ──► Your loop ──► Claude  (decides)
             ▲             │
             │             ▼  "use this tool"
        send result     Your tools ──► real systems (inventory, payments)
             │             │
             └─────────────┘  (loop until Claude says "done")
```

- **Claude** decides what to do. It never touches your systems.
- **Your tools** are the only way to actually do anything — so that's where you add safety checks.
- For risky actions (refunds, payments), add a **human approval** step before your code runs the tool.


---
# 🚀 Mini Project — Zepto Restock Helper

**The task:** each morning, make a restock plan — which items are low, how many to order to reach the target, and the total cost. A human approves before any real order goes out.

**Build it (reuse what you already wrote):**
1. Tools: `check_stock`, `reorder_amount`, and `current_price(item)`.
2. Run the same loop from Part 4.
3. Use **parallel** for the per-item lookups, then **chain** into the totals.
4. Before any real order, print it and ask a human "ok? y/n".

**Done when:** it lists the items, the amounts, and the total — and never places a real order without the human saying yes.


---
# 📝 Quick Summary
- A chatbot **talks**; an agent **acts** using tools and reacts to results.
- The loop: **look → think → act → check → repeat**, steered by `stop_reason`.
- **Claude decides, your code runs the tool**, then sends the result back.
- A tool = **name + description + input_schema**; the **description** picks the tool.
- The **`messages` list is the memory** — always send the tool result back.
- **Chain** tools when they depend on each other; **parallel** when they don't.
- 4 bugs: never stops · wrong tool · made-up inputs · ignores result → print the trace to find it.

# 🧠 Cheat Sheet
| Word | Plain meaning |
|---|---|
| Agent | AI that uses tools to *do* things |
| Tool | a function you let Claude use (name + description + input_schema) |
| `stop_reason` | why Claude stopped: `tool_use` (keep going) or `end_turn` (done) |
| `tool_result` | the answer your code sends back after running a tool |
| Memory | the `messages` list you keep adding to and resending |
| Chaining | one tool's answer feeds the next tool |

**The core API call (memorise the shape):**
```python
reply = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)
# reply.stop_reason → "tool_use" or "end_turn"
# reply.content[i].type → "text" or "tool_use" (then .name, .input, .id)
```

# ⏱️ 5-Minute Revision
1. Chatbot describes, agent does. Claude decides, your code acts.
2. Loop = look → think → act → check; `stop_reason` steers it.
3. Tool = name + description + input_schema; description picks the tool.
4. `messages` is the memory — always send the tool result back.
5. Chain dependent tools; parallel for independent ones.
6. Bug? Print every step.

# 🎯 Interview Questions (with answers)
1. *Chatbot vs agent?* → A chatbot writes text; an agent uses tools to act and reacts to results.
2. *Who runs the tool?* → Your code. Claude only asks.
3. *Which field steers the loop?* → `stop_reason` (`tool_use` = keep going, `end_turn` = stop).
4. *Why might an agent loop forever?* → The tool result wasn't sent back.
5. *Fix wrong-tool choice?* → Make the descriptions clearer.

# 🧾 Mini Assessment
1. Who runs a tool — Claude or your code?
2. What are the three parts of a tool?
3. While `stop_reason` is ___, keep looping.
4. Where does the agent keep its memory?
5. Chain or parallel: looking up 3 unrelated prices at once?

**Answers:** 1) Your code. 2) name, description, input_schema. 3) `"tool_use"`. 4) The `messages` list. 5) Parallel.

---
## 🔍 Learn more (official docs)
- Tool use — https://docs.claude.com/en/docs/build-with-claude/tool-use
- Messages API — https://docs.claude.com/en/api/messages
